# Prompt Caching for Open-Weight LLMs on Amazon SageMaker — End-to-End

**Companion notebook to the blog/paper "Prompt Caching Without the Vendor."**

This notebook demonstrates **engine-level prompt caching** for an open-weight model
(`DeepSeek-R1-Distill-Qwen-1.5B`) on Amazon SageMaker, entirely from **SageMaker Studio**.

Instead of running an inference engine inside the Studio kernel (Studio disables local Docker by
default), we use the **SageMaker-native pattern**: deploy the model to a **real-time endpoint using
the AWS Large Model Inference (LMI) container**. LMI's **vLLM** engine performs *automatic prefix
caching* — the same engine-level KV-reuse mechanism as SGLang's RadixAttention discussed in the
paper. We then measure **cold (cache miss) vs. warm (cache hit)** request latency against the
endpoint and translate it into a cost model.

### What it does
1. Deploy `DeepSeek-R1-Distill-Qwen-1.5B` on an `ml.g5.xlarge` endpoint (LMI + vLLM, prefix caching on).
2. Benchmark cold vs. warm latency across a range of shared-prefix lengths.
3. Plot the results and compute $/1M-request savings.
4. **Delete the endpoint** (important — it bills per hour while running).

### Prerequisites
- Run in **SageMaker Studio** (JupyterLab). The kernel instance can be small (e.g., `ml.t3.medium`) —
  the GPU work happens on the **endpoint**, not the kernel.
- Execution role with SageMaker + ECR + (optional) Hugging Face access.
- Account quota for **`ml.g5.xlarge` for endpoint usage** in your Region.
- Uses **SageMaker Python SDK v2** (the setup cell pins `sagemaker<3`, since v3 is a breaking rewrite).




## 1. Setup

In [ ]:
# Pin to SageMaker SDK v2 — v3.x is a major API rewrite that removes image_uris/Model/get_execution_role.
%pip install -q "sagemaker>=2.240,<3" boto3
import sagemaker, boto3, json, time, uuid
from sagemaker import image_uris

sess = sagemaker.Session()
region = sess.boto_region_name
import os
try:
    role = os.environ.get("SM_EXEC_ROLE") or sagemaker.get_execution_role()
except Exception:
    # Fallback: set SM_EXEC_ROLE env var, or paste your SageMaker execution role ARN here.
    role = os.environ.get("SM_EXEC_ROLE", "arn:aws:iam::<ACCOUNT_ID>:role/<SageMakerExecutionRole>")
print("region:", region)
print("role:", role)
print("sagemaker:", sagemaker.__version__)


## 2. Configuration

In [ ]:
MODEL_ID       = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"  # try -7B on ml.g5.2xlarge
INSTANCE_TYPE  = "ml.g5.xlarge"
ENABLE_PREFIX_CACHING = True     # the feature under test
MAX_MODEL_LEN  = 16384

# Workload
VARIABLE_TOKENS = 48
MAX_NEW_TOKENS  = 16             # small fixed decode so latency is prefill-dominated
PREFIX_SWEEP    = [512, 1024, 2048, 4096, 8192]   # words in the shared prefix
N_REPEAT        = 6              # warm repetitions (median reported)

# Cost model (edit to your Region's SageMaker hosting price)
ENDPOINT_HOURLY_USD = 1.408      # ml.g5.xlarge real-time hosting (us-east-1) — VERIFY for your Region

ENDPOINT_NAME = f"pc-poc-{uuid.uuid4().hex[:8]}"
print("endpoint will be:", ENDPOINT_NAME)


## 3. Resolve the LMI container image

In [ ]:
# LMI ("djl-lmi") is the AWS Large Model Inference container. Retrieve the latest image for the Region.
try:
    inference_image = image_uris.retrieve(framework="djl-lmi", region=region, version="latest")
except Exception as e:
    print("could not resolve 'latest', trying a pinned version:", e)
    inference_image = image_uris.retrieve(framework="djl-lmi", region=region, version="0.33.0")
print(inference_image)


## 4. Deploy the endpoint (LMI + vLLM, prefix caching enabled)
This step takes **~8–15 minutes** (image pull + model download + warmup).

In [ ]:
from sagemaker.model import Model

env = {
    "HF_MODEL_ID": MODEL_ID,
    "OPTION_ROLLING_BATCH": "vllm",
    "OPTION_ENABLE_PREFIX_CACHING": "true" if ENABLE_PREFIX_CACHING else "false",
    "OPTION_MAX_MODEL_LEN": str(MAX_MODEL_LEN),
    "OPTION_DTYPE": "bf16",
    "OPTION_TENSOR_PARALLEL_DEGREE": "1",
    "OPTION_TRUST_REMOTE_CODE": "true",
}

model = Model(image_uri=inference_image, env=env, role=role, sagemaker_session=sess)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
    container_startup_health_check_timeout=1200,
)
print("endpoint InService:", ENDPOINT_NAME)


## 5. Invocation helper and prompt builder
We measure end-to-end request latency for a small fixed decode, so the delta is dominated by prefill — which is exactly what prefix caching eliminates on a warm hit.

In [ ]:
rt = boto3.client("sagemaker-runtime", region_name=region)

_WORD = "policy"
def build_prompt(prefix_words, unique_tag):
    # Stable, cacheable prefix FIRST; unique/variable content LAST.
    prefix = (unique_tag + " ") + " ".join([_WORD] * prefix_words)
    variable = " ".join([f"detail{i}" for i in range(VARIABLE_TOKENS)])
    return prefix + "\n\nUser query: " + variable

def invoke_latency(prompt):
    payload = {"inputs": prompt,
               "parameters": {"max_new_tokens": MAX_NEW_TOKENS, "temperature": 0.0,
                              "do_sample": False}}
    t0 = time.perf_counter()
    resp = rt.invoke_endpoint(EndpointName=ENDPOINT_NAME,
                              ContentType="application/json",
                              Body=json.dumps(payload))
    _ = resp["Body"].read()
    return (time.perf_counter() - t0) * 1000.0  # ms

def measure(prefix_words, n=N_REPEAT):
    # Unique tag makes the first call a guaranteed cache MISS (cold); repeats are warm HITS.
    tag = uuid.uuid4().hex
    prompt = build_prompt(prefix_words, tag)
    cold = invoke_latency(prompt)
    warm = sorted(invoke_latency(prompt) for _ in range(n))[n // 2]  # median
    return {"prefix_words": prefix_words, "latency_cold_ms": round(cold, 1),
            "latency_warm_ms": round(warm, 1),
            "reduction_pct": round((1 - warm / cold) * 100, 1)}

# warmup (loads CUDA graphs / first batch)
_ = invoke_latency(build_prompt(256, "warmup"))
print("warmup ok")


## 6. Run the benchmark

In [ ]:
import pandas as pd

rows = [measure(pw) for pw in PREFIX_SWEEP]
for r in rows:
    print(r)
df = pd.DataFrame(rows)
df.to_csv("studio_results.csv", index=False)
df


## 7. Plot: cold vs. warm latency

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
x = range(len(df)); w = 0.35
ax[0].bar([i - w/2 for i in x], df["latency_cold_ms"], w, label="cold (miss)")
ax[0].bar([i + w/2 for i in x], df["latency_warm_ms"], w, label="warm (hit)")
ax[0].set_xticks(list(x)); ax[0].set_xticklabels(df["prefix_words"])
ax[0].set_xlabel("shared prefix (words)"); ax[0].set_ylabel("latency (ms)")
ax[0].set_title("Request latency: cold vs warm"); ax[0].legend()

ax[1].plot(df["prefix_words"], df["reduction_pct"], marker="o")
ax[1].set_xlabel("shared prefix (words)"); ax[1].set_ylabel("latency reduction (%)")
ax[1].set_title("Prefix-caching benefit vs prefix length"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig("studio_prefix_caching.png", dpi=150); plt.show()


## 8. Cost model

In [ ]:
import numpy as np

def cost_per_1m(hit_rate, price_hr=ENDPOINT_HOURLY_USD, row=None):
    gsc = row["latency_cold_ms"] / 1000.0
    gsw = row["latency_warm_ms"] / 1000.0
    per_req = price_hr / 3600.0 * (gsw * hit_rate + gsc * (1 - hit_rate))
    return per_req * 1e6

biggest = df.sort_values("prefix_words").iloc[-1]
hit = np.linspace(0, 1, 11)
plt.figure(figsize=(7, 4))
plt.plot(hit * 100, [cost_per_1m(h, row=biggest) for h in hit], marker="o")
plt.xlabel("cache-hit rate (%)"); plt.ylabel("cost per 1M requests (USD)")
plt.title(f"Cost vs hit-rate @ {biggest['prefix_words']}-word prefix"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("studio_cost.png", dpi=150); plt.show()

print(f"All-cold  $/1M: {cost_per_1m(0.0, row=biggest):.2f}")
print(f"80% hit   $/1M: {cost_per_1m(0.8, row=biggest):.2f}")


## 9. Cleanup — **run this to stop billing**

In [ ]:
# Delete endpoint, endpoint config, and model to stop hourly charges.
try:
    predictor.delete_endpoint(delete_endpoint_config=True)
    print("endpoint + config deleted")
except Exception as e:
    print("predictor delete issue, falling back to boto3:", e)
    sm = boto3.client("sagemaker", region_name=region)
    for fn, kw in [(sm.delete_endpoint, {"EndpointName": ENDPOINT_NAME}),
                   (sm.delete_endpoint_config, {"EndpointConfigName": ENDPOINT_NAME})]:
        try: fn(**kw)
        except Exception as e2: print("skip:", e2)
try:
    model.delete_model(); print("model deleted")
except Exception as e:
    print("model delete:", e)


## 10. Notes 
- This endpoint path uses **vLLM automatic prefix caching** inside the LMI container — the same
  engine-level KV-reuse concept as SGLang RadixAttention (which the paper benchmarks directly on EC2).
- Endpoint measurements include network + queueing overhead, so absolute ms differ from the
  bare-metal SGLang numbers; the **trend** (warm ≪ cold, benefit rising with prefix length) is the
  publishable result.
- To show caching **on vs. off**, re-run with `ENABLE_PREFIX_CACHING = False` on a fresh endpoint
  name and compare — warm should then ≈ cold.
